# Modulo AI — classificazione del materiale dei capi (Capitolo 4.3)

Notebook per **Google Colab** (gratuito). Esegui le celle in ordine.

1. Menu *Runtime → Cambia tipo di runtime → GPU (T4)*.
2. Il dataset è **TextileNet-fibre** (Zhong et al., 2023, licenza CC BY): immagini di capi etichettate per fibra.
3. Il modello è una **MobileNetV3-Large** pre-addestrata su ImageNet, riaddestrata sulle 9 classi di materiale della piattaforma (transfer learning).
4. Alla fine scarichi `materiali.onnx`, `classi.json`, `metriche.json` e `matrice_confusione.png` da copiare in `ai-module/models/`.

Le metriche finali (accuratezza, F1 macro, matrice di confusione) vanno riportate nel Capitolo 5.

In [ ]:
!pip -q install gdown onnx onnxruntime
!nvidia-smi -L || echo 'Nessuna GPU: l\'addestramento sarà lento'

## 1. Scarica TextileNet-fibre
Link ufficiale dal repository https://github.com/hahashu/TextileNet (Google Drive).

In [ ]:
!mkdir -p data/raw
!gdown 1e_E9NeTs7qSuUzWszSkmK09jTHPQwdd6 -O data/raw/textilenet_fibre.zip
!cd data/raw && unzip -q -o textilenet_fibre.zip
import pathlib
candidati = [p.parent for p in pathlib.Path('data/raw').rglob('cotton') if p.is_dir()]
SORGENTE = candidati[0]
print('Cartella con le fibre:', SORGENTE)
print(sorted(p.name for p in SORGENTE.iterdir() if p.is_dir()))

## 2. Codice (identico ai file in `ai-module/src/`)

In [ ]:
%%writefile classi.py
"""Classi di materiale riconosciute dal modello e corrispondenza con le etichette di TextileNet.

TextileNet (Zhong et al., 2023) etichetta le immagini per FIBRA (33 classi).
Le fibre sono raggruppate nelle 9 classi di materiale usate dalla piattaforma
(le stesse del campo "materialePrincipale" del backend).
"""

CLASSI = ["cotone", "lana", "cashmere", "seta", "lino", "pelle", "poliestere", "nylon", "viscosa"]

# etichetta TextileNet-fibre -> classe della piattaforma (le altre fibre sono escluse)
MAPPA_TEXTILENET = {
    "cotton": "cotone",
    "wool": "lana",
    "alpaca": "lana",
    "camel": "lana",
    "llama": "lana",
    "mohair": "lana",
    "yak": "lana",
    "angora": "lana",
    "cashmere": "cashmere",
    "silk": "seta",
    "flax_linen": "lino",
    "leather": "pelle",
    "suede": "pelle",
    "polyester": "poliestere",
    "nylon": "nylon",
    "viscose_rayon": "viscosa",
    "modal": "viscosa",
    "lyocell": "viscosa",
    "cupro": "viscosa",
}

# Normalizzazione ImageNet: la stessa usata dalla rete pre-addestrata
MEDIA = [0.485, 0.456, 0.406]
DEVIAZIONE = [0.229, 0.224, 0.225]
LATO = 224


In [ ]:
%%writefile prepara_dataset.py
"""Prepara il dataset per l'addestramento a partire da TextileNet-fibre.

Legge data/raw/fibre/<etichetta_textilenet>/*.jpg, raggruppa le fibre nelle 9
classi della piattaforma e crea data/processed/{train,val,test}/<classe>/
(70% / 15% / 15%, divisione stratificata e riproducibile).

Uso:  python src/prepara_dataset.py --sorgente data/raw/fibre --destinazione data/processed --max-per-classe 1500
"""
import argparse
import json
import random
import shutil
from collections import Counter
from pathlib import Path

from classi import CLASSI, MAPPA_TEXTILENET

ESTENSIONI = {".jpg", ".jpeg", ".png", ".webp"}


def raccogli(sorgente: Path) -> dict[str, list[Path]]:
    per_classe: dict[str, list[Path]] = {c: [] for c in CLASSI}
    for cartella in sorted(p for p in sorgente.iterdir() if p.is_dir()):
        classe = MAPPA_TEXTILENET.get(cartella.name)
        if classe is None:
            continue
        per_classe[classe].extend(sorted(f for f in cartella.rglob("*") if f.suffix.lower() in ESTENSIONI))
    return per_classe


def dividi(file: list[Path], seme: int, quote=(0.7, 0.15)) -> dict[str, list[Path]]:
    file = file[:]
    random.Random(seme).shuffle(file)
    n = len(file)
    n_train, n_val = int(n * quote[0]), int(n * quote[1])
    return {"train": file[:n_train], "val": file[n_train : n_train + n_val], "test": file[n_train + n_val :]}


def prepara(sorgente: Path, destinazione: Path, max_per_classe: int, seme: int = 42) -> dict:
    per_classe = raccogli(sorgente)
    if destinazione.exists():
        shutil.rmtree(destinazione)
    conteggi = {"train": Counter(), "val": Counter(), "test": Counter()}
    for classe, file in per_classe.items():
        random.Random(seme).shuffle(file)
        for parte, elenco in dividi(file[:max_per_classe], seme).items():
            conteggi[parte][classe] = len(elenco)
            if not elenco:
                continue
            cartella = destinazione / parte / classe
            cartella.mkdir(parents=True, exist_ok=True)
            for i, f in enumerate(elenco):
                shutil.copy2(f, cartella / f"{classe}_{i:05d}{f.suffix.lower()}")
    riepilogo = {parte: dict(c) for parte, c in conteggi.items()}
    (destinazione / "riepilogo.json").write_text(json.dumps(riepilogo, indent=2, ensure_ascii=False))
    return riepilogo


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--sorgente", default="data/raw/fibre")
    ap.add_argument("--destinazione", default="data/processed")
    ap.add_argument("--max-per-classe", type=int, default=1500)
    ap.add_argument("--seme", type=int, default=42)
    a = ap.parse_args()
    r = prepara(Path(a.sorgente), Path(a.destinazione), a.max_per_classe, a.seme)
    for parte, c in r.items():
        print(f"{parte:6s} {sum(c.values()):6d} immagini  {c}")


In [ ]:
%%writefile addestra.py
"""Addestramento del classificatore di materiali (transfer learning) ed esportazione ONNX.

Rete: MobileNetV3-Large pre-addestrata su ImageNet (torchvision), ultimo strato
sostituito con 9 uscite (le classi di materiale). Due fasi:
  1) solo il nuovo classificatore (rete congelata)       -> apprende in fretta
  2) sblocco degli ultimi blocchi con learning rate basso -> affina le feature
Metriche sul test set: accuratezza, F1 macro, report per classe, matrice di confusione.
Il modello viene esportato in ONNX: il servizio di inferenza usa onnxruntime e non
richiede PyTorch.

Uso (Colab con GPU consigliato):  python src/addestra.py --dati data/processed --uscita models
"""
import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

from classi import CLASSI, DEVIAZIONE, LATO, MEDIA


def trasformazioni():
    addestramento = transforms.Compose([
        transforms.RandomResizedCrop(LATO, scale=(0.6, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.15, contrast=0.15),  # lieve: il colore è informativo
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, DEVIAZIONE),
    ])
    valutazione = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(LATO),
        transforms.ToTensor(),
        transforms.Normalize(MEDIA, DEVIAZIONE),
    ])
    return addestramento, valutazione


def carica_dati(cartella: Path, lotto: int):
    t_train, t_eval = trasformazioni()
    insiemi = {
        "train": datasets.ImageFolder(cartella / "train", t_train),
        "val": datasets.ImageFolder(cartella / "val", t_eval),
        "test": datasets.ImageFolder(cartella / "test", t_eval),
    }
    for nome, ds in insiemi.items():
        assert ds.classes == sorted(CLASSI), f"classi inattese in {nome}: {ds.classes}"
    caricatori = {n: DataLoader(ds, batch_size=lotto, shuffle=(n == "train"), num_workers=2) for n, ds in insiemi.items()}
    return insiemi, caricatori


def crea_modello(n_classi: int) -> nn.Module:
    modello = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
    for p in modello.parameters():
        p.requires_grad = False
    ingresso = modello.classifier[-1].in_features
    modello.classifier[-1] = nn.Linear(ingresso, n_classi)
    return modello


def sblocca_ultimi_blocchi(modello: nn.Module, n: int = 4):
    for blocco in list(modello.features.children())[-n:]:
        for p in blocco.parameters():
            p.requires_grad = True


def pesi_classi(ds) -> torch.Tensor:
    conteggi = np.bincount(ds.targets, minlength=len(ds.classes)).astype(np.float32)
    pesi = conteggi.sum() / (len(conteggi) * np.maximum(conteggi, 1))
    return torch.tensor(pesi, dtype=torch.float32)


def epoca(modello, caricatore, criterio, ottimizzatore, dispositivo):
    addestra = ottimizzatore is not None
    modello.train(addestra)
    totale, corretti, perdita = 0, 0, 0.0
    with torch.set_grad_enabled(addestra):
        for x, y in caricatore:
            x, y = x.to(dispositivo), y.to(dispositivo)
            uscita = modello(x)
            loss = criterio(uscita, y)
            if addestra:
                ottimizzatore.zero_grad()
                loss.backward()
                ottimizzatore.step()
            perdita += loss.item() * len(y)
            corretti += (uscita.argmax(1) == y).sum().item()
            totale += len(y)
    return perdita / totale, corretti / totale


def valuta(modello, caricatore, dispositivo, n_classi):
    modello.eval()
    matrice = np.zeros((n_classi, n_classi), dtype=int)
    with torch.no_grad():
        for x, y in caricatore:
            previsti = modello(x.to(dispositivo)).argmax(1).cpu().numpy()
            for vero, prev in zip(y.numpy(), previsti):
                matrice[vero, prev] += 1
    precisione = np.diag(matrice) / np.maximum(matrice.sum(0), 1)
    richiamo = np.diag(matrice) / np.maximum(matrice.sum(1), 1)
    f1 = 2 * precisione * richiamo / np.maximum(precisione + richiamo, 1e-9)
    return {
        "accuratezza": float(np.trace(matrice) / matrice.sum()),
        "f1_macro": float(f1.mean()),
        "per_classe": {
            c: {"precisione": float(p), "richiamo": float(r), "f1": float(f), "campioni": int(n)}
            for c, p, r, f, n in zip(sorted(CLASSI), precisione, richiamo, f1, matrice.sum(1))
        },
        "matrice_confusione": matrice.tolist(),
    }


def salva_matrice_png(matrice, classi, percorso: Path):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    m = np.array(matrice)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(m, cmap="Greens")
    ax.set_xticks(range(len(classi)), classi, rotation=45, ha="right")
    ax.set_yticks(range(len(classi)), classi)
    ax.set_xlabel("Classe prevista")
    ax.set_ylabel("Classe reale")
    for i in range(len(classi)):
        for j in range(len(classi)):
            ax.text(j, i, m[i, j], ha="center", va="center", fontsize=8, color="black" if m[i, j] < m.max() / 2 else "white")
    ax.set_title("Matrice di confusione (test set)")
    fig.tight_layout()
    fig.savefig(percorso, dpi=160)


def esporta_onnx(modello, percorso: Path, dispositivo):
    modello.eval().to("cpu")
    esempio = torch.randn(1, 3, LATO, LATO)
    torch.onnx.export(
        modello, esempio, str(percorso), input_names=["immagine"], output_names=["punteggi"],
        dynamic_axes={"immagine": {0: "lotto"}, "punteggi": {0: "lotto"}}, opset_version=17,
    )
    modello.to(dispositivo)
    try:  # controllo: onnxruntime deve dare lo stesso risultato di PyTorch
        import onnxruntime as ort

        sessione = ort.InferenceSession(str(percorso), providers=["CPUExecutionProvider"])
        atteso = modello.to("cpu")(esempio).detach().numpy()
        ottenuto = sessione.run(None, {"immagine": esempio.numpy()})[0]
        print(f"Verifica ONNX: differenza massima {np.abs(atteso - ottenuto).max():.2e}")
        modello.to(dispositivo)
    except ImportError:
        print("onnxruntime non installato: verifica ONNX saltata")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dati", default="data/processed")
    ap.add_argument("--uscita", default="models")
    ap.add_argument("--epoche-testa", type=int, default=3)
    ap.add_argument("--epoche-affinamento", type=int, default=5)
    ap.add_argument("--lotto", type=int, default=64)
    a = ap.parse_args()

    torch.manual_seed(42)
    dispositivo = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Dispositivo: {dispositivo}")
    uscita = Path(a.uscita)
    uscita.mkdir(parents=True, exist_ok=True)

    insiemi, caricatori = carica_dati(Path(a.dati), a.lotto)
    classi = insiemi["train"].classes
    modello = crea_modello(len(classi)).to(dispositivo)
    criterio = nn.CrossEntropyLoss(weight=pesi_classi(insiemi["train"]).to(dispositivo))

    storia, migliore, inizio = [], 0.0, time.time()
    fasi = [("testa", a.epoche_testa, 1e-3), ("affinamento", a.epoche_affinamento, 1e-4)]
    for fase, n_epoche, lr in fasi:
        if fase == "affinamento":
            sblocca_ultimi_blocchi(modello)
        ottimizzatore = torch.optim.AdamW((p for p in modello.parameters() if p.requires_grad), lr=lr, weight_decay=1e-4)
        for e in range(n_epoche):
            l_tr, a_tr = epoca(modello, caricatori["train"], criterio, ottimizzatore, dispositivo)
            l_va, a_va = epoca(modello, caricatori["val"], criterio, None, dispositivo)
            storia.append({"fase": fase, "epoca": e + 1, "loss_train": l_tr, "acc_train": a_tr, "loss_val": l_va, "acc_val": a_va})
            print(f"[{fase}] epoca {e + 1}/{n_epoche}  acc train {a_tr:.3f}  acc val {a_va:.3f}")
            if a_va > migliore:
                migliore = a_va
                torch.save(modello.state_dict(), uscita / "migliore.pt")

    modello.load_state_dict(torch.load(uscita / "migliore.pt", map_location=dispositivo))
    metriche = valuta(modello, caricatori["test"], dispositivo, len(classi))
    metriche.update({"classi": classi, "storia": storia, "durata_s": round(time.time() - inizio), "rete": "mobilenet_v3_large (ImageNet)", "immagini": {k: len(v) for k, v in insiemi.items()}})
    (uscita / "metriche.json").write_text(json.dumps(metriche, indent=2, ensure_ascii=False))
    (uscita / "classi.json").write_text(json.dumps(classi, ensure_ascii=False))
    salva_matrice_png(metriche["matrice_confusione"], classi, uscita / "matrice_confusione.png")
    esporta_onnx(modello, uscita / "materiali.onnx", dispositivo)
    print(f"\nTest set: accuratezza {metriche['accuratezza']:.3f}, F1 macro {metriche['f1_macro']:.3f}")
    print(f"File salvati in {uscita.resolve()}: materiali.onnx, classi.json, metriche.json, matrice_confusione.png")


if __name__ == "__main__":
    main()


## 3. Prepara il dataset (70% train, 15% validazione, 15% test)

In [ ]:
!python prepara_dataset.py --sorgente "$SORGENTE" --destinazione data/processed --max-per-classe 1500

## 4. Addestramento ed esportazione ONNX
Con GPU T4 richiede indicativamente alcune decine di minuti.

In [ ]:
!python addestra.py --dati data/processed --uscita models --epoche-testa 3 --epoche-affinamento 5

## 5. Risultati

In [ ]:
import json
from IPython.display import Image, display
m = json.load(open('models/metriche.json'))
print(f"Accuratezza test: {m['accuratezza']:.3f} | F1 macro: {m['f1_macro']:.3f}")
for c, v in m['per_classe'].items():
    print(f"{c:12s} precisione {v['precisione']:.2f}  richiamo {v['richiamo']:.2f}  F1 {v['f1']:.2f}  ({v['campioni']} immagini)")
display(Image('models/matrice_confusione.png'))

## 6. Scarica i file del modello
Copiali nella cartella `ai-module/models/` del progetto.

In [ ]:
from google.colab import files
for f in ['materiali.onnx', 'classi.json', 'metriche.json', 'matrice_confusione.png']:
    files.download(f'models/{f}')